In [ ]:
#
# You can copy this entire code block into a single cell in your Jupyter Notebook.
#

import numpy as np
import matplotlib.pyplot as plt

# =================================================================================
#  FUNCTION 1: For creating a simple, single pulse sequence
# =================================================================================
def create_simple_sequence(pulse_sequence_steps, channel_mapping):
    """
    Processes a user-defined sequence and returns the low-level data and total period.
    This is ideal for creating a single, fixed sequence.

    Args:
        pulse_sequence_steps (list): A list of dictionaries defining the pulse sequence.
        channel_mapping (dict): A map of channel names to physical channel numbers.

    Returns:
        tuple: (low_level_sequence, total_period_us)
    """
    print("--- Creating simple sequence ---")
    total_period_us = sum(step['duration_us'] for step in pulse_sequence_steps)
    
    low_level_sequence = []
    for step in pulse_sequence_steps:
        duration_ns = int(step['duration_us'] * 1000)
        # Safely get channel list, defaulting to empty if not provided
        channel_names_on = step.get('channels_on', [])
        high_channels = [channel_mapping.get(name) for name in channel_names_on]
        
        if duration_ns > 0:
            low_level_sequence.append((duration_ns, high_channels, 0.0, 0.0))
            
    print(f"Total sequence duration: {total_period_us:.2f} µs")
    return low_level_sequence, total_period_us

# =================================================================================
#  FUNCTION 2: For creating a continuous sweep sequence
# =================================================================================
def create_sweep_sequence(sweep_params, subsequence_generator, channel_mapping):
    """
    Generates one single, continuous sequence for a sweep experiment.

    Args:
        sweep_params (dict): A dictionary with 'start', 'end', and 'steps' for the sweep.
        subsequence_generator (function): A function that takes a sweep value
                                          and returns the pulse steps for one sub-sequence.
        channel_mapping (dict): A map of channel names to physical channel numbers.

    Returns:
        tuple: (full_low_level_sequence, total_duration_us)
    """
    print(f"--- Creating sweep sequence with {sweep_params['steps']} steps ---")
    sweep_values = np.linspace(sweep_params['start'], sweep_params['end'], sweep_params['steps'])
    
    full_sequence_data = []
    
    for value in sweep_values:
        # 1. Generate the sub-sequence for the current sweep value
        sub_sequence_steps = subsequence_generator(value)
        
        # 2. Convert the sub-sequence to low-level format and append it
        for step in sub_sequence_steps:
            duration_ns = int(step['duration_us'] * 1000)
            channel_names_on = step.get('channels_on', [])
            high_channels = [channel_mapping.get(name) for name in channel_names_on]
            
            if duration_ns > 0:
                full_sequence_data.append((duration_ns, high_channels, 0.0, 0.0))

    total_duration_ns = sum(p[0] for p in full_sequence_data)
    total_duration_us = total_duration_ns / 1000.0
    
    print(f"Total continuous sequence duration: {total_duration_us:.2f} µs")
    return full_sequence_data, total_duration_us


# =================================================================================
#  PLOTTING FUNCTION (Adapted to plot a portion of a long sequence)
# =================================================================================
def plot_sequence(low_level_sequence, channel_mapping, plot_duration_us=None, title='Pulse Sequence'):
    """
    Generates a plot for a low-level pulse sequence. Can plot a partial duration.
    """
    plot_data = low_level_sequence
    
    # If a plot duration is specified, truncate the data for plotting
    if plot_duration_us is not None:
        plot_duration_ns = plot_duration_us * 1000
        plot_data = []
        current_time_ns = 0
        for pulse in low_level_sequence:
            plot_data.append(pulse)
            current_time_ns += pulse[0]
            if current_time_ns >= plot_duration_ns:
                break
    
    print(f"--- Generating Plot: {title} ---")
    fig, ax = plt.subplots(figsize=(15, 6))
    channel_names = {v: k for k, v in channel_mapping.items()}
    active_channels = sorted(list(channel_mapping.values()))

    for chan_num in active_channels:
        x_coords, y_coords = [0], [0]
        current_time_ns = 0
        for duration_ns, high_channels, _, _ in plot_data:
            level = 1 if chan_num in high_channels else 0
            if y_coords[-1] != level:
                x_coords.append(current_time_ns)
                y_coords.append(level)
            current_time_ns += duration_ns
            x_coords.append(current_time_ns)
            y_coords.append(level)
        
        time_us = np.array(x_coords) / 1000.0
        waveform = np.array(y_coords) * 0.8 + (chan_num - 1)
        ax.plot(time_us, waveform, label=channel_names.get(chan_num, f'CH {chan_num}'))

    ax.set_title(title, fontsize=16)
    ax.set_xlabel('Time (µs)', fontsize=12)
    ax.set_ylabel('Digital Channel', fontsize=12)
    ax.set_yticks([ch - 1 for ch in active_channels])
    ax.set_yticklabels([channel_names.get(ch, f'CH {ch}') for ch in active_channels])
    ax.set_ylim(-0.5, 8)
    ax.grid(axis='y', linestyle=':')
    ax.legend()
    plt.show()

In [ ]:
# --- Define hardware channels ---
channel_map = {
    'Rabi_Pulse': 1,
    'Trigger': 5,
}

In [ ]:
# Define the steps for a single, fixed sequence
my_fixed_steps = [
    {'duration_us': 50, 'channels_on': ['Rabi_Pulse', 'Trigger']},
    {'duration_us': 200, 'channels_on': []},
    {'duration_us': 200, 'channels_on': ['Rabi_Pulse']},
]

# Create and plot the simple sequence
simple_data, simple_period = create_simple_sequence(my_fixed_steps, channel_map)
plot_sequence(simple_data, channel_map, title="My Fixed Sequence")

In [ ]:
# --- 1. Define Sweep Parameters ---
rabi_sweep_params = {
    'start': 2,    # Start sweep with a 2 µs Rabi pulse
    'end': 22,     # End with a 22 µs pulse
    'steps': 5,    # Perform 5 steps in the sweep
}

# Fixed wait time after each Rabi pulse
WAIT_TIME_US = 50
# Fixed duration for the trigger that starts each step
TRIGGER_DURATION_US = 0.05 # 50 ns

# --- 2. Define the Sub-Sequence Generator ---
# This function defines the structure of ONE step in the sweep.
# It takes the current Rabi pulse duration as input.
def rabi_subsequence_generator(rabi_duration_us):
    
    pulse_after_trigger_us = rabi_duration_us - TRIGGER_DURATION_US
    if pulse_after_trigger_us < 0:
        pulse_after_trigger_us = 0
    
    # This list of dictionaries is the "recipe" for one step
    sub_sequence = [
        # Part 1: Trigger and Rabi Pulse ON
        {'duration_us': TRIGGER_DURATION_US, 'channels_on': ['Rabi_Pulse', 'Trigger']},
        # Part 2: Rabi Pulse continues alone
        {'duration_us': pulse_after_trigger_us, 'channels_on': ['Rabi_Pulse']},
        # Part 3: Wait time for measurement
        {'duration_us': WAIT_TIME_US, 'channels_on': []}
    ]
    return sub_sequence

# --- 3. Create the full, continuous sweep sequence ---
full_rabi_sequence, total_rabi_duration = create_sweep_sequence(
    rabi_sweep_params,
    rabi_subsequence_generator,
    channel_map
)

# --- 4. Plot the first few steps for verification ---
# We calculate the duration of the first 3 steps to set a sensible plot window
one_step_approx_duration = rabi_sweep_params['start'] + WAIT_TIME_US
plot_window = one_step_approx_duration * 3

plot_sequence(
    full_rabi_sequence,
    channel_map,
    plot_duration_us=plot_window,
    title="Rabi Sweep Sequence (First 3 Steps)"
)